# Lesson 5.2: Sentiment Analysis with Hugging Face (RoBERTa)

**🎬 Video:** [Lesson 5.2: Sentiment Analysis with Hugging Face (RoBERTa)](#)

## Overview

In Lesson 5.1 you used **VADER**, a fast rule-based tool, to score sentiment. In this lesson you will use **RoBERTa**, a transformer-based language model fine-tuned on Twitter data, and compare its results to VADER. 

By the end of this lesson you will have:
- Loaded and run the `cardiffnlp/twitter-roberta-base-sentiment` model from Hugging Face
- Applied it to the same geoparsed Reddit sentences from Lesson 5.1

RoBERTa (Robustly Optimized BERT Pretraining Approach) is available on [Hugging Face](https://huggingface.co/). Check out other sentiment models [here](https://huggingface.co/models?sort=trending&search=sentiment).

---

## 1 Running RoBERTa

While running VADER was not particurally complicated RoBERTa requires a bit more finesse. Because this requires knowing concepts outside the scope of this course and some helper functions, all the more complicated logic has been tucked into `sentiment_utils.py`. In the code below, we are first going to recreate our Vader Sentiment scores from the previous lesson and then we are going to create a sample column of RoBERTa scores. Because RoBERTA takes a long time to run, you want to avoid running it constantly. Indeed, for the final project you should only run it if you find that some of the locations are off and need to be fixed

In [ ]:
import os
import pandas as pd
from sentiment_utils import add_sentiment_to_column
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

nltk.download("vader_lexicon", quiet=True)
sia = SentimentIntensityAnalyzer()

_primary = "../data/JMU/JMU_geoparsed_cleaned.pickle"
_backup = "../data/JMU/JMU_geoparsed_cleaned_backup.csv"

if os.path.exists(_primary):
    df_reddit_geoparsed_long = pd.read_pickle(_primary)
    print(f"✅ Loaded from JMU_geoparsed_cleaned.pickle ({len(df_reddit_geoparsed_long):,} rows)")
else:
    df_reddit_geoparsed_long = pd.read_csv(_backup)
    print(f"⚠️  JMU_geoparsed_cleaned.pickle not found — using backup ({len(df_reddit_geoparsed_long):,} rows)")
    print("   Complete Lesson 4.4 to use your team's cleaned data.")

df_reddit_geoparsed_long["vader_sentiment"] = df_reddit_geoparsed_long["sentences"].apply(
    lambda x: sia.polarity_scores(str(x))["compound"]
)
print(f"✅ Sentiment scored — {len(df_reddit_geoparsed_long):,} rows total")

df_reddit_sentiment_sample = add_sentiment_to_column(
    df_reddit_geoparsed_long, "sentences", num_rows=100
)

_scored = df_reddit_sentiment_sample["roberta_compound"].notna().sum()
print(f"✅ RoBERTa scored {_scored:,} / {len(df_reddit_sentiment_sample):,} sample rows")
if _scored == 0:
    raise RuntimeError(
        "RoBERTa scoring failed for every row. Restart the kernel, re-run this cell, "
        "and check for a warning starting with 'RoBERTa scoring failed'."
    )


In [ ]:
df_reddit_sentiment_sample.head()

### Evaluate the Sample

While the analysis only tagged the emotions for 100 sentences this should be enough to see the differences between the two taggers. The output below looks at where the two taggers disagree and pulls a random sample.


In [ ]:
RANDOM_STATE = 41  # ← change this to see different contradictions

# Filter to rows where VADER and RoBERTa disagree significantly, then sample
df_reddit_sentiment_sample["disagreement"] = (
    df_reddit_sentiment_sample["vader_sentiment"] - df_reddit_sentiment_sample["roberta_compound"]
).abs()

contradictions = df_reddit_sentiment_sample[df_reddit_sentiment_sample["disagreement"] > 0.3]

(
    contradictions[["sentences", "vader_sentiment", "roberta_compound"]]
    .sample(4, random_state=RANDOM_STATE)
    .style
    .set_properties(subset=["sentences"], **{"white-space": "normal", "width": "500px"})
    .format({"vader_sentiment": "{:.3f}", "roberta_compound": "{:.3f}"})
)

> 💡 **Critical Reflection:**
> 1. How did the models do? 
> 2. Which appears to be more accurate? Why do you think that is?
> 3. Are there sentences where you do not think either emotional evaluation is accurate?
> Cycle through the samples by changing `random_state=` to a different integer and identify a sentence where the language model does particularly well or poorly.

---

## 2 Creating the Full Dataset

Running RoBERTa on all sentences takes a long time and only **one** team member should do it. 

1. **Create a new branch** — e.g. `firstname-sentiment-data`
2. Run the cell below to create the DataFrame and save files (approx. 3-5 minutes)
2. Open **Source Control** (<img src="../lesson_assets/images/vscode/source-control.svg" alt="Source Control icon" width="14">) and confirm only `data/jmu_reddit_sentiment_full.pickle` appears under **Changes**
   - If any other file appears, click the **Discard Changes** icon next to it
3. Stage the file (`+`), then use the ✨ sparkle button to auto-generate a commit message
4. Click the dropdown next to **Commit** and select **Commit & Sync**
5. Click **Publish Branch**, then open a Pull Request with base `main`
6. Merge the PR and delete the branch
7. **Everyone else:** switch back to `main` and click **Sync Changes** to download the file


The line below is intentionally **commented out**. When you are ready, one group member should remove the `#`, run this cell, then complete Section 7 to save and share the result.


In [ ]:
# Create DataFrame with sentiment scores for all rows, then save to pickle
import os
import pandas as pd
from sentiment_utils import add_sentiment_to_column

df_reddit_sentiment_full = add_sentiment_to_column(df_reddit_geoparsed_long, "sentences", output_format="compact")
df_reddit_sentiment_full.to_pickle("../data/JMU/JMU_geoparsed_cleaned_sentiment.pickle")
df_reddit_sentiment_full.to_csv("../data/JMU/JMU_geoparsed_cleaned_sentiment.csv")
print(
    f"✅ Saved {len(df_reddit_sentiment_full):,} rows to data/JMU/JMU_geoparsed_cleaned_sentiment.pickle"
)

### Verify Completion

Once everyone is synced up, verify that the process worked by sampling the data.


In [ ]:
import pandas as pd

try:
        df_reddit_sentiment_full = pd.read_pickle("../data/JMU/JMU_geoparsed_cleaned_sentiment.pickle")
        print(f"✅ Loaded {len(df_reddit_sentiment_full):,} rows")
except FileNotFoundError:
        try:
            df_reddit_sentiment_full = pd.read_pickle(
                "../data/JMU/JMU_geoparsed_cleaned_sentiment_backup.pickle"
            )
            print(f"⚠️  Teammate file not found — loaded instructor backup ({len(df_reddit_sentiment_full):,} rows)")
            print("   This is pre-scored data from the instructor's backup CSV.")
            print("   Your group still needs to run and commit the full dataset (Part 2) to replace this.")
        except FileNotFoundError:
            print("❌ Neither the team file nor the instructor backup was found.")
            print("   Ask the teammate who ran the full analysis to complete Section 7 first.")


In [ ]:
(
    df_reddit_sentiment_full[["sentences", "place", "roberta_compound"]]
    .sample(n=5, random_state=42)
    .style
    .set_properties(subset=["sentences"], **{"white-space": "normal", "width": "700px"})
    .set_properties(subset=["place"], **{"white-space": "normal", "width": "180px"})
    .set_properties(subset=["roberta_compound"], **{"width": "120px", "text-align": "center"})
    .format({"roberta_compound": "{:.3f}"})
)

---

## Lesson Summary

Here is what you covered in this lesson:

### Overview: RoBERTa — Transformer-Based Sentiment
- **RoBERTa** — a transformer model fine-tuned on Twitter data; understands context across the whole sentence rather than word by word
- **`AutoTokenizer` / `AutoModelForSequenceClassification`** — Hugging Face classes for loading pre-trained models
- **`softmax()`** — converts raw model scores into probabilities that sum to 1
- **`roberta_compound`** — a derived score: `(pos - neg) × (1 - neu)`, ranging from −1 to +1

### Part 1: Comparing Methods
- **Contradictions** — sentences where VADER and RoBERTa disagree most reveal each model's blind spots
- **Edge cases** — comparing outputs at the extremes exposes the assumptions built into each model

### Part 2: Running the model
- Run the sentiment model on your data
- The **branch → PR workflow** from Lesson 1.1 ensures only one clean version of the data lands in `main`

---

➡️ **Next:** [Lesson 6 — Mapping Fundamentals](../lesson_6_mapping_fundamentals/lesson_6_mapping_fundamentals.ipynb)
